<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/LangStudio/RSI_Recent_Detail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import yfinance as yf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

### Remember to load the OptionVolume.csv file before running the scripts

In [2]:
# ==========================================
# ⚙️ GLOBAL CONFIGURATION (TUNING)
# ==========================================
# Format: (Length, Threshold)
RSI_CONFIG = [
    (16, 25),
    (24, 30),
    (14, 25),
    (22, 30),
    (18, 25),
    (22, 25),
    (26, 30)
]
LOOKBACK_DAYS = 14      # How far back to search for trigger events
CSV_FILE = "OptionVolume200.csv"  # Source for your ticker universe
FALLBACK_SYMBOLS = ["TSLA", "NVDA", "AAPL", "AMD", "MSFT", "BTC-USD"]

# ==========================================
# 📈 PRECISION CALCULATION LOGIC
# ==========================================

def calculate_rsi_precision(series, period=14):
    """Matches Yahoo Finance / Wilder's Smoothing Precision"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # com=period-1 is the mathematical equivalent to Wilder's Smoothing
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def scan_for_triggers():
    """Main scanning engine for recent RSI events"""
    # 1. Load Symbols
    try:
        df_csv = pd.read_csv(CSV_FILE)
        symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower()][0]
        symbols = df_csv[symbol_col].str.strip().unique().tolist()
        print(f"✅ Loaded {len(symbols)} symbols from {CSV_FILE}")
    except:
        symbols = FALLBACK_SYMBOLS
        print(f"⚠️ {CSV_FILE} not found. Using fallback list: {symbols}")

    report_data = []

    # Calculate dates
    # Fetch 300 extra days to allow RSI math to stabilize (Wilder's Warm-up)
    start_fetch = (datetime.now() - timedelta(days=LOOKBACK_DAYS + 300)).strftime('%Y-%m-%d')
    trigger_cutoff = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).date()

    print(f"🔍 Scanning for RSI triggers using {len(RSI_CONFIG)} configurations since {trigger_cutoff}...")

    for symbol in symbols:
        try:
            # Fetch data once per symbol
            df = yf.download(symbol, start=start_fetch, progress=False, auto_adjust=True)
            if df.empty: continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            # NEW: Loop through each Length/Threshold pair
            for length, threshold in RSI_CONFIG:
                # Calculate RSI for this specific length
                rsi_col = f'RSI_{length}'
                df[rsi_col] = calculate_rsi_precision(df['Close'], period=length)

                # TRIGGER: Check against the specific threshold for this length
                trigger_col = f'Trigger_{length}'
                df[trigger_col] = (df[rsi_col] < threshold) & (df[rsi_col].shift(1) >= threshold)

                # Filter for the lookback window
                recent_hits = df[df.index.date >= trigger_cutoff]
                recent_hits = recent_hits[recent_hits[trigger_col] == True]

                for date, row in recent_hits.iterrows():
                    report_data.append({
                        "Date": date.strftime('%Y-%m-%d'),
                        "Symbol": symbol,
                        "Price": round(row['Close'], 2),
                        "RSI_Len": length,           # Added to track which pair triggered
                        "RSI_Val": round(row[rsi_col], 2),
                        "Thresh": threshold          # Added for clarity
                    })
        except Exception as e:
            print(f"❌ Error processing {symbol}: {e}")
            continue

    return report_data

# ==========================================
# 📝 EXECUTION & OUTPUT GENERATION
# ==========================================

if __name__ == "__main__":
    triggers = scan_for_triggers()

    print("\n" + "="*70)
    print(f"🚀 RSI MULTI-PAIR REPORT: LAST {LOOKBACK_DAYS} DAYS")
    print(f"Pairs Scanned: {RSI_CONFIG}")
    print("="*70)

    if not triggers:
        print(f"No triggers found in the last {LOOKBACK_DAYS} days.")
    else:
        results_df = pd.DataFrame(triggers)
        # Sort by Date (newest) then Symbol
        results_df = results_df.sort_values(by=["Date", "Symbol"], ascending=[False, True])
        print(results_df.to_string(index=False))

    print("="*70)

✅ Loaded 200 symbols from OptionVolume200.csv
🔍 Scanning for RSI triggers using 7 configurations since 2026-04-24...

🚀 RSI MULTI-PAIR REPORT: LAST 14 DAYS
Pairs Scanned: [(16, 25), (24, 30), (14, 25), (22, 30), (18, 25), (22, 25), (26, 30)]
      Date Symbol  Price  RSI_Len  RSI_Val  Thresh
2026-05-08    ABT  84.42       16    24.38      25
2026-05-08    ABT  84.42       14    23.48      25
2026-05-08    ABT  84.42       26    27.79      30
2026-05-08    MCD 276.10       22    29.99      30
2026-05-08   SQQQ  42.84       24    28.81      30
2026-05-08   SQQQ  42.84       18    24.35      25
2026-05-08   SQQQ  42.84       26    29.97      30
2026-05-06    ABT  86.30       14    24.13      25
2026-05-06   SQQQ  45.59       16    24.52      25
2026-05-06   SQQQ  45.59       14    22.46      25
2026-05-06   SQQQ  45.59       22    29.32      30
2026-05-05    LMT 508.93       14    24.49      25
2026-05-04    ABT  87.54       24    29.08      30
2026-05-04    ABT  87.54       22    28.53  

In [3]:
# ==========================================
# ⚙️ GLOBAL CONFIGURATION (MOMENTUM TUNE)
# ==========================================
# Format: (Length, Threshold)
RSI_CONFIG = [
    (10, 80),
    (14, 75)
]
LOOKBACK_DAYS = 14      # Lookback window for trigger events
SMA_TREND = 200         # Baseline trend filter
CSV_FILE = "OptionVolume.csv"
FALLBACK_SYMBOLS = ["TSLA", "NVDA", "AAPL", "AMD", "MSFT", "COIN", "MSTR"]

# ==========================================
# 📈 PRECISION CALCULATION LOGIC
# ==========================================

def calculate_rsi_precision(series, period=14):
    """Matches Wilder's Smoothing Precision (Yahoo Style)"""
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    # com=period-1 is equivalent to Wilder's alpha=1/period
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()

    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def scan_for_momentum_triggers():
    """Identifies 'Momentum Ignition' (RSI crossing ABOVE threshold)"""
    try:
        df_csv = pd.read_csv(CSV_FILE)
        symbol_col = [c for c in df_csv.columns if 'symbol' in c.lower()][0]
        symbols = df_csv[symbol_col].str.strip().unique().tolist()
    except:
        symbols = FALLBACK_SYMBOLS

    report_data = []

    # Fetch 350 days to ensure SMA 200 and RSI 10 are both stable
    start_fetch = (datetime.now() - timedelta(days=LOOKBACK_DAYS + 350)).strftime('%Y-%m-%d')
    trigger_cutoff = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).date()

    print(f"🚀 Scanning for Momentum Ignition (RSI configurations: {RSI_CONFIG}) since {trigger_cutoff}...")

    for symbol in symbols:
        try:
            df = yf.download(symbol, start=start_fetch, progress=False, auto_adjust=True)
            if df.empty or len(df) < SMA_TREND: continue

            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)

            df['SMA_200'] = df['Close'].rolling(window=SMA_TREND).mean()
            df['Trend'] = np.where(df['Close'] > df['SMA_200'], "UP", "DOWN")

            # NEW: Evaluate each specific RSI pair
            for length, threshold in RSI_CONFIG:
                rsi_vals = calculate_rsi_precision(df['Close'], period=length)

                # TRIGGER: Today is above threshold, Yesterday was below (Cross Above)
                trigger = (rsi_vals > threshold) & (rsi_vals.shift(1) <= threshold)

                # Filter for the lookback window
                recent_hits = df[df.index.date >= trigger_cutoff].copy()
                recent_hits['Is_Trigger'] = trigger
                hits = recent_hits[recent_hits['Is_Trigger'] == True]

                for date, row in hits.iterrows():
                    report_data.append({
                        "Date": date.strftime('%Y-%m-%d'),
                        "Symbol": symbol,
                        "Price": f"${row['Close']:.2f}",
                        "RSI_Len": length,           # Track which length triggered
                        "RSI_Val": round(rsi_vals.loc[date], 1),
                        "Thresh": threshold,         # Track the target threshold
                        "Trend": row['Trend']
                    })
        except Exception:
            continue

    return report_data

# ==========================================
# 📝 EXECUTION & OUTPUT GENERATION
# ==========================================

if __name__ == "__main__":
    triggers = scan_for_momentum_triggers()

    print("\n" + "="*80)
    print(f"🚀 RSI MOMENTUM POWER ZONE REPORT (LAST {LOOKBACK_DAYS} DAYS)")
    print(f"Pairs Scanned: {RSI_CONFIG}")
    print("="*80)

    if not triggers:
        print(f"No momentum breakouts detected in the last {LOOKBACK_DAYS} days.")
    else:
        results_df = pd.DataFrame(triggers)
        # Sort by Date (newest first)
        results_df = results_df.sort_values(by="Date", ascending=False)

        print(results_df.to_string(index=False))

    print("="*80)


🚀 Scanning for Momentum Ignition (RSI configurations: [(10, 80), (14, 75)]) since 2026-04-24...

🚀 RSI MOMENTUM POWER ZONE REPORT (LAST 14 DAYS)
Pairs Scanned: [(10, 80), (14, 75)]
      Date Symbol    Price  RSI_Len  RSI_Val  Thresh Trend
2026-05-08    AMD  $449.28       10     81.2      80    UP
2026-05-08   SNDK $1549.01       10     81.6      80    UP
2026-05-08   PANW  $205.44       14     75.2      75    UP
2026-05-08   PANW  $205.44       10     80.4      80    UP
2026-05-08    NVO   $46.08       14     75.3      75  DOWN
2026-05-08    NVO   $46.08       10     80.7      80  DOWN
2026-05-08    SMH  $565.17       10     82.5      80    UP
2026-05-08    SPY  $737.22       14     75.3      75    UP
2026-05-08     MU  $739.78       10     86.4      80    UP
2026-05-08    WDC  $474.69       14     75.5      75    UP
2026-05-08   SNDK $1549.01       14     79.7      75    UP
2026-05-08   SOXL  $175.82       14     77.2      75    UP
2026-05-08   AMZN  $273.04       14     75.3      75